# Foto de capa vs. review textual — amostra estratificada

**Pergunta:** olhando lado a lado a capa e o que os hóspedes escreveram, anúncios bem e mal
avaliados se distinguem pela foto? E nos anúncios em que os hóspedes dizem que o lugar **não
bate com as fotos**, o que a capa mostra?

**Amostra:** 200 anúncios, `src.amostra_reviews.selecionar_amostra` — 4 grupos × 2 tipos × 25.

| grupo | critério | |
|---|---|---|
| `accuracy_baixa` | `review_scores_accuracy` < 4,5 | "o lugar corresponde às fotos?" — tem prioridade |
| `nota_baixa` | nota < 4,5 | os ~3 % piores |
| `nota_media` | 4,7 ≤ nota < 4,85 | |
| `nota_alta` | nota ≥ 4,95 | |

Filtros: ≥ 20 reviews, último review desde 2025 (a capa é a de hoje; review antigo descreveria
outro estado do imóvel), foto disponível. Tipos: casa/apto inteiro e quarto privativo, separados
porque a capa de quarto costuma mostrar a área comum.

**As notas são relativas.** Com ≥ 20 reviews, a mediana de SP é 4,88 e só 11 anúncios ficam
abaixo de 4,0 — "nota baixa" aqui é ruim *para o Airbnb paulistano*, não em termos absolutos.
Em `nota_baixa × quarto` só existem 21 anúncios; os 4 que faltam vieram de `inteiro`.

**Saídas**
- `data/interim/amostra_reviews.csv` — os 200 anúncios e o grupo de cada um
- `data/interim/amostra_reviews_ids.txt` — os mesmos ids, um por linha (entrada do Street View)
- `data/raw/Airbnb/photos_amostra/{id}.jpg` — capas (pasta separada de `photos/`, que define o piloto)
- `data/processed/Airbnb/amostra_reviews_textos.csv` — por anúncio: review mais negativo, mais
  positivo e o que mais fala das fotos
- `outputs/foto_vs_review_<grupo>.png`, `outputs/foto_vs_review_clip.png`

## 1. Setup

In [ ]:
import json
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import stats

sys.path.insert(0, "../..")  # raiz do repo (o notebook roda de notebooks/imageAnalysis/)
from src.airbnb_photos import baixar_capas  # noqa: E402
from src.amostra_reviews import FOTOS_AMOSTRA, GRUPOS, selecionar_amostra  # noqa: E402

ROOT = Path("../..")
PROC = ROOT / "data/processed/Airbnb"
INTERIM = ROOT / "data/interim"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

SEED = 42
DESDE = "2025-01-01"
MAX_REVIEWS = 30  # por anúncio, os mais recentes

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Amostra e capas

Seleciona, baixa as capas e, se alguma voltar com erro (foto trocada ou removida depois do
scrape), exclui o anúncio e seleciona de novo. A exclusão só puxa o próximo da fila do mesmo
grupo e tipo — os outros anúncios não mudam. Idempotente: capa já baixada não é baixada de
novo. Sem API, sem custo: as fotos vêm do CDN do Airbnb.

In [ ]:
falhas = set()
for rodada in range(1, 6):
    amostra = selecionar_amostra(desde=DESDE, seed=SEED, excluir=falhas)
    rel = baixar_capas(amostra.id, listings=amostra, destino=FOTOS_AMOSTRA)
    novas = set(rel.loc[~rel.status.isin(["ok", "existente"]), "id"])
    print(f"rodada {rodada}: {len(amostra)} anúncios, {len(novas)} capas com erro")
    if not novas:
        break
    falhas |= novas

assert not novas, "ainda há capas com erro depois de 5 rodadas"
print(f"excluídos por capa indisponível: {sorted(falhas)}")
amostra.to_csv(INTERIM / "amostra_reviews.csv", index=False)
# um id por linha: entrada de scripts/fetch_streetview.py --ids-arquivo
(INTERIM / "amostra_reviews_ids.txt").write_text("
".join(map(str, amostra.id)) + "
")

print(f"{len(amostra)} anúncios, {amostra.neighbourhood_cleansed.nunique()} bairros")
print(pd.crosstab(amostra.grupo, amostra.tipo, margins=True))
amostra.groupby("grupo", observed=True)[
    ["review_scores_rating", "review_scores_accuracy", "review_scores_cleanliness", "number_of_reviews"]
].median().round(2)

## 3. Reviews

Os `MAX_REVIEWS` mais recentes de cada anúncio, desde `DESDE`. `reviews_clean.csv` já vem
filtrado por `text_processing.ipynb` (≥ 20 caracteres e ≥ 3 palavras, sem mensagens automáticas).
O nome do hóspede não é usado.

In [ ]:
ids = set(amostra.id)
partes = []
for ch in pd.read_csv(PROC / "reviews_clean.csv", usecols=["listing_id", "date", "comments"],
                      chunksize=300_000, low_memory=False):
    partes.append(ch[ch.listing_id.isin(ids) & (ch.date >= DESDE)])

rev = (pd.concat(partes)
         .sort_values(["listing_id", "date"], ascending=[True, False])
         .groupby("listing_id").head(MAX_REVIEWS)
         .reset_index(drop=True))

n = rev.listing_id.value_counts()
print(f"{len(rev):,} reviews de {rev.listing_id.nunique()} anúncios")
print(f"por anúncio: mediana {n.median():.0f}, mínimo {n.min()}")

## 4. Polaridade do texto

Mesmo modelo e mesma lógica de protótipos de `image_review_match.ipynb`: o encoder multilíngue
põe português, inglês e espanhol no mesmo espaço, então protótipos em português cobrem todos os
reviews. Dois eixos, por review:

- **geral** — `polaridade = cos(review, positivo) − cos(review, negativo)`
- **fotos** — o review fala de o lugar bater (ou não) com as fotos? `relevância = max(cos+, cos−)`

In [ ]:
from transformers import AutoModel, AutoTokenizer

MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
tok = AutoTokenizer.from_pretrained(MODEL)
mod = AutoModel.from_pretrained(MODEL).to(device).eval()


def encode(texts, bs=128, max_len=128):
    '''Mean pooling mascarado + L2 — igual a image_review_match.ipynb.'''
    out = []
    for i in range(0, len(texts), bs):
        b = tok(texts[i:i + bs], padding=True, truncation=True,
                max_length=max_len, return_tensors="pt").to(device)
        with torch.no_grad():
            h = mod(**b).last_hidden_state
        m = b["attention_mask"].unsqueeze(-1).float()
        v = (h * m).sum(1) / m.sum(1)
        out.append((v / v.norm(dim=-1, keepdim=True)).cpu().numpy())
    return np.vstack(out)


PROTO = {
    "geral": (["a estadia foi ótima, recomendo muito",
               "lugar excelente, confortável e anfitrião atencioso"],
              ["a estadia foi péssima, não recomendo",
               "tive muitos problemas, o lugar era ruim e desconfortável"]),
    "fotos": (["o apartamento é exatamente como nas fotos",
               "tudo igual ao anúncio, as fotos são fiéis"],
              ["o lugar é bem diferente das fotos",
               "as fotos enganam, o apartamento é pior do que parece"]),
}

R = encode(rev.comments.tolist())
for eixo, (pos, neg) in PROTO.items():
    cp = (R @ encode(pos).T).mean(1)
    cn = (R @ encode(neg).T).mean(1)
    rev[f"{eixo}_pol"] = cp - cn
    rev[f"{eixo}_rel"] = np.maximum(cp, cn)

rev[["geral_pol", "fotos_pol", "fotos_rel"]].describe().round(3)

**Sanidade:** a polaridade média dos reviews de cada anúncio precisa acompanhar a nota. Se não
acompanhar, o eixo não mede o que diz medir e a galeria abaixo mostraria reviews errados.

In [ ]:
por_anuncio = rev.groupby("listing_id").agg(
    pol_media=("geral_pol", "mean"),
    neg_share=("geral_pol", lambda s: (s < 0).mean()),
    n_reviews_txt=("geral_pol", "size"),
)
amostra = amostra.merge(por_anuncio, left_on="id", right_index=True, how="left")

for col in ["pol_media", "neg_share"]:
    r, p = stats.spearmanr(amostra[col], amostra.review_scores_rating, nan_policy="omit")
    print(f"{col:>9} vs nota:  rho = {r:+.2f}  (p = {p:.1e})")

amostra.groupby("grupo", observed=True)[["pol_media", "neg_share"]].median().round(3)

### Reviews representativos

Por anúncio: o mais negativo, o mais positivo e — entre os que falam de fotos (relevância no
top 10 % do eixo) — o mais negativo nesse eixo. Vai para um CSV, para ler os 200 com calma.

In [ ]:
lim_fotos = rev.fotos_rel.quantile(0.90)


def extremo(g, col, fn):
    return g.loc[fn(g[col]), "comments"] if len(g) else None


linhas = []
for lid, g in rev.groupby("listing_id"):
    gf = g[g.fotos_rel >= lim_fotos]
    linhas.append({
        "id": lid,
        "review_mais_negativo": extremo(g, "geral_pol", pd.Series.idxmin),
        "review_mais_positivo": extremo(g, "geral_pol", pd.Series.idxmax),
        "review_sobre_fotos": extremo(gf, "fotos_pol", pd.Series.idxmin),
        "fotos_pol_min": gf.fotos_pol.min() if len(gf) else np.nan,
    })
textos = pd.DataFrame(linhas)

cols = ["id", "grupo", "tipo", "name", "neighbourhood_cleansed", "price", "review_scores_rating",
        "review_scores_accuracy", "number_of_reviews", "pol_media", "neg_share", "listing_url"]
saida = amostra[cols].merge(textos, on="id", how="left")
saida["foto"] = saida.id.map(lambda i: str(FOTOS_AMOSTRA / f"{i}.jpg"))
saida.to_csv(PROC / "amostra_reviews_textos.csv", index=False)
print(saida.shape)
print("anúncios com algum review falando de fotos:", saida.review_sobre_fotos.notna().sum())

## 5. Galeria — foto ao lado do review

Por grupo, 5 anúncios de cada tipo. Abaixo da foto: nota, accuracy, nº de reviews e o review
**mais negativo** (o mais informativo para ver o que a foto não mostra). `galeria(grupo, seed=...)`
sorteia outros 10; a lista completa está no CSV.

In [ ]:
def corta(s, n=230):
    s = str(s)
    return s if len(s) <= n else s[:n].rsplit(" ", 1)[0] + " …"


def galeria(grupo, k=5, seed=SEED, coluna="review_mais_negativo", salvar=True):
    sub = saida[saida.grupo == grupo]
    sel = pd.concat([g.sample(min(k, len(g)), random_state=seed) for _, g in sub.groupby("tipo")])
    fig, axes = plt.subplots(2, k, figsize=(4.2 * k, 11.5))
    for ax in axes.ravel():
        ax.axis("off")
    for row, (tipo, g) in enumerate(sel.groupby("tipo")):
        for col, r in enumerate(g.itertuples()):
            ax = axes[row, col]
            ax.imshow(Image.open(r.foto))
            ax.set_title(f"{r.id} · {tipo}\nnota {r.review_scores_rating:.2f} · accuracy "
                         f"{r.review_scores_accuracy:.2f} · {r.number_of_reviews} reviews", fontsize=9)
            txt = getattr(r, coluna)
            ax.text(0, -0.04, textwrap.fill(corta(txt), 52) if isinstance(txt, str) else "(sem review)",
                    transform=ax.transAxes, va="top", fontsize=7.5, wrap=True)
    fig.suptitle(f"{grupo} — {coluna.replace('_', ' ')}", fontsize=13, y=0.995)
    fig.tight_layout(h_pad=9)
    if salvar:
        fig.savefig(OUT / f"foto_vs_review_{grupo}.png", dpi=100, bbox_inches="tight")
    plt.show()

In [ ]:
galeria("accuracy_baixa")

In [ ]:
galeria("nota_baixa")

In [ ]:
galeria("nota_media")

In [ ]:
galeria("nota_alta")

Para o grupo `accuracy_baixa`, o review que fala de fotos costuma ser mais revelador que o mais
negativo em geral:

In [ ]:
galeria("accuracy_baixa", coluna="review_sobre_fotos", salvar=False)

## 6. A foto "parece" diferente entre os grupos?

Os mesmos eixos zero-shot do CLIP de `image_features.ipynb` (prompts lidos de
`clip_prompts.json`, para serem idênticos): margem `cos(foto, positivo) − cos(foto, negativo)`
por aspecto. Kruskal-Wallis por aspecto entre os 4 grupos, com correção de Benjamini-Hochberg.

Com ~50 fotos por grupo, só diferenças grandes aparecem. Ausência de diferença não prova que
as fotos são iguais.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

prm = json.load(open(PROC / "clip_prompts.json"))
clip = CLIPModel.from_pretrained(prm["model"]).to(device).eval()
cproc = CLIPProcessor.from_pretrained(prm["model"])


def norm(out):
    t = out if torch.is_tensor(out) else out.pooler_output  # transformers <5 vs >=5
    return t / t.norm(dim=-1, keepdim=True)


aspectos = prm["aspects"]
with torch.no_grad():
    temb = {a: [norm(clip.get_text_features(**cproc(text=lado, return_tensors="pt",
                                                    padding=True).to(device)))
                for lado in (pos, neg)]
            for a, (pos, neg) in aspectos.items()}

feats = []
for i in range(0, len(saida), 32):
    lote = saida.iloc[i:i + 32]
    ims = [Image.open(p).convert("RGB") for p in lote.foto]
    with torch.no_grad():
        e = norm(clip.get_image_features(**cproc(images=ims, return_tensors="pt").to(device)))
    d = {"id": lote.id.to_numpy()}
    for a, (tp, tn) in temb.items():
        d[f"img_{a}"] = ((e @ tp.T).mean(1) - (e @ tn.T).mean(1)).float().cpu().numpy()
    feats.append(pd.DataFrame(d))

img = pd.concat(feats).merge(saida[["id", "grupo", "tipo"]], on="id")
img.groupby("grupo", observed=True)[[f"img_{a}" for a in aspectos]].median().round(3)

In [ ]:
def bh(p):
    p = np.asarray(p)
    o = np.argsort(p)
    adj = np.minimum.accumulate((p[o] * len(p) / np.arange(1, len(p) + 1))[::-1])[::-1]
    out = np.empty_like(adj)
    out[o] = adj.clip(max=1)
    return out


res = []
for a in aspectos:
    amostras = [img.loc[img.grupo == g, f"img_{a}"] for g in GRUPOS]
    h, p = stats.kruskal(*amostras)
    res.append({"aspecto": a, "H": h, "p": p})
res = pd.DataFrame(res)
res["p_bh"] = bh(res.p)
res["sig"] = res.p_bh < 0.05
print(res.round(4).to_string(index=False))

fig, axes = plt.subplots(1, len(aspectos), figsize=(3.2 * len(aspectos), 3.6), sharey=False)
cores = ["#b2182b", "#ef8a62", "#999999", "#2166ac"]
for ax, a in zip(axes, aspectos):
    dados = [img.loc[img.grupo == g, f"img_{a}"] for g in GRUPOS]
    bp = ax.boxplot(dados, patch_artist=True, widths=0.6)
    for patch, c in zip(bp["boxes"], cores):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    ax.set_xticks(range(1, 5), ["acc↓", "nota↓", "média", "nota↑"], fontsize=8)
    sig = res.set_index("aspecto").loc[a, "p_bh"]
    ax.set_title(f"{a}\np_bh = {sig:.3f}", fontsize=9)
fig.suptitle("Eixos CLIP da capa por grupo", fontsize=11)
fig.tight_layout()
fig.savefig(OUT / "foto_vs_review_clip.png", dpi=110, bbox_inches="tight")
plt.show()

## 7. Leitura

*Preencher depois de olhar as galerias.* Perguntas-guia:

1. Em `accuracy_baixa`, a capa é mais "produzida" (luz, decoração, ângulo amplo) do que em
   `nota_baixa`? O eixo CLIP de `decor`/`light` aponta nessa direção?
2. O que os reviews negativos de `nota_alta` reclamam? Coisas que uma foto não mostra
   (barulho, check-in, anfitrião) ou coisas visíveis (limpeza, estado do imóvel)?
3. Em quarto privativo, a capa mostra o quarto ou a área comum? Isso muda entre grupos?